In [1]:
import pandas as pd
import numpy as np

# =========================================================
# Параметры
# =========================================================
PATH = r"C:\Users\Ксения\Downloads\Тестовое_задание_Data_аналитик_ЦО_2025_.xlsx"

SHEET_DATA = "Данные для задачи 1"
SHEET_MARKUP = "Справочник наценок"

ANALYSIS_DATE = pd.Timestamp.today().normalize()  # дата, на которую "активны" правила
TARGET_DELTA_RUB = 1                              # хотим быть на 1 руб ниже минимума рынка
MIN_MARGIN = 0.10                                 # маржинальность >= 10% ( (P - C) / P )

COMPETITORS = [
    "letu.ru", "iledebeaute.ru", "rivegauche.ru", "Ozon", "Wildberries",
    "MagnitMarket", "Megamarket", "lamoda.ru", "tsum.ru", "visagehall.ru"
]

In [2]:
# =========================================================
# Вспомогательные функции
# =========================================================
def to_lower_str(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip().str.lower()


def filter_active_rules(rules: pd.DataFrame, asof: pd.Timestamp) -> pd.DataFrame:
    """Оставляем только правила:
    - Активный = 'Да'
    - asof попадает в [Дата начала, Дата окончания] (если даты не пустые)
    """
    r = rules.copy()

    r["Дата начала"] = pd.to_datetime(r["Дата начала"], errors="coerce")
    r["Дата окончания"] = pd.to_datetime(r["Дата окончания"], errors="coerce")
    r["Активный"] = to_lower_str(r["Активный"])

    r = r[r["Активный"].eq("да")]
    r = r[
        (r["Дата начала"].isna() | (r["Дата начала"] <= asof))
        & (r["Дата окончания"].isna() | (r["Дата окончания"] >= asof))
    ]
    return r


# Порядок приоритетов по ТЗ
PRIORITY_LEVELS = [
    ("product", None),
    ("brand_cat3", "Категория3"),
    ("brand_cat2", "Категория2"),
    ("brand_cat1", "Категория1"),
    ("brand", None),
    ("seg_cat3", "Категория3"),
    ("seg_cat2", "Категория2"),
    ("seg_cat1", "Категория1"),
    ("seg", None),
]


def pick_markup_for_row(row: pd.Series, rules_active: pd.DataFrame) -> tuple[float, str | None]:
    """Возвращает (наценка, уровень_правила) по дереву приоритетов.
    Если ничего не найдено — (NaN, None).
    """
    code = int(row["Код товара"])
    brand = row["Бренд"]
    seg = row["Сегмент"]

    # Продукт в справочнике может быть float/str -> приводим к Int64 один раз
    # (делаем это снаружи, но оставлю защиту здесь на случай изменений)
    if "Продукт_int" not in rules_active.columns:
        rules_active = rules_active.copy()
        rules_active["Продукт_int"] = pd.to_numeric(rules_active["Продукт"], errors="coerce").round().astype("Int64")

    for level, cat_field in PRIORITY_LEVELS:
        if level == "product":
            cand = rules_active[
                rules_active["Продукт_int"].notna() & (rules_active["Продукт_int"] == code)
            ]

        elif level == "brand":
            cand = rules_active[
                rules_active["Бренд"].notna()
                & rules_active["Категория"].isna()
                & rules_active["Сегмент рынка"].isna()
                & (rules_active["Бренд"] == brand)
            ]

        elif level.startswith("brand_cat"):
            cat_val = row[cat_field]
            cand = rules_active[
                rules_active["Бренд"].notna()
                & rules_active["Категория"].notna()
                & rules_active["Сегмент рынка"].isna()
                & (rules_active["Бренд"] == brand)
                & (rules_active["Категория"] == cat_val)
            ]

        elif level == "seg":
            cand = rules_active[
                rules_active["Сегмент рынка"].notna()
                & rules_active["Категория"].isna()
                & rules_active["Бренд"].isna()
                & (rules_active["Сегмент рынка"] == seg)
            ]

        elif level.startswith("seg_cat"):
            cat_val = row[cat_field]
            cand = rules_active[
                rules_active["Сегмент рынка"].notna()
                & rules_active["Категория"].notna()
                & rules_active["Бренд"].isna()
                & (rules_active["Сегмент рынка"] == seg)
                & (rules_active["Категория"] == cat_val)
            ]

        else:
            cand = rules_active.iloc[0:0]

        if len(cand) > 0:
            # Если совпало несколько правил — берём с самой поздней датой начала (как самое "свежее")
            cand = cand.sort_values("Дата начала", ascending=False)
            return float(cand.iloc[0]["Наценка"]), level

    return np.nan, None


def safe_row_min_and_argmin(df_prices: pd.DataFrame) -> tuple[pd.Series, pd.Series]:
    """Безопасно считает минимум по строке и имя колонки-минимума.
    Если в строке всё NaN — минимум = NaN, argmin = NaN (а не исключение).
    """
    row_min = df_prices.min(axis=1, skipna=True)

    # argmin через idxmin падает на all-NaN, поэтому делаем маску
    all_nan = df_prices.isna().all(axis=1)
    argmin = pd.Series(np.nan, index=df_prices.index, dtype="object")
    if (~all_nan).any():
        argmin.loc[~all_nan] = df_prices.loc[~all_nan].idxmin(axis=1, skipna=True)

    return row_min, argmin

In [3]:
# =========================================================
# 1) Загрузка данных
# =========================================================
df = pd.read_excel(PATH, sheet_name=SHEET_DATA)
rules = pd.read_excel(PATH, sheet_name=SHEET_MARKUP)

# Минимальные проверки структуры (чтобы не ловить "тихие" ошибки)
required_df_cols = {"Бренд", "Код товара", "Категория1", "Категория2", "Категория3", "Сегмент", "Цена закупки товара"}
required_rules_cols = {"Сегмент рынка", "Категория", "Бренд", "Продукт", "Дата начала", "Дата окончания", "Наценка", "Активный"}

missing_df = required_df_cols - set(df.columns)
missing_rules = required_rules_cols - set(rules.columns)
missing_comp = set(COMPETITORS) - set(df.columns)

if missing_df:
    raise ValueError(f"В листе '{SHEET_DATA}' отсутствуют колонки: {sorted(missing_df)}")
if missing_rules:
    raise ValueError(f"В листе '{SHEET_MARKUP}' отсутствуют колонки: {sorted(missing_rules)}")
if missing_comp:
    raise ValueError(f"В листе '{SHEET_DATA}' отсутствуют колонки конкурентов: {sorted(missing_comp)}")

In [4]:
# =========================================================
# 2) Наценка (по приоритетам, с учётом активности/дат)
# =========================================================
rules_active = filter_active_rules(rules, ANALYSIS_DATE)
rules_active["Продукт_int"] = pd.to_numeric(rules_active["Продукт"], errors="coerce").round().astype("Int64")

picked = df.apply(lambda r: pick_markup_for_row(r, rules_active), axis=1, result_type="expand")
picked.columns = ["Наценка", "Уровень_правила"]
df = df.join(picked)

# Цена до скидки: закупка * (1 + наценка)
df["Цена_до_скидки"] = df["Цена закупки товара"] * (1 + df["Наценка"])


# =========================================================
# 3) Минимальная цена конкурента + кто дал минимум
# =========================================================
comp_prices = df[COMPETITORS].copy()
df["Мин_цена_конкурента"], df["Кто_минимум"] = safe_row_min_and_argmin(comp_prices)


# =========================================================
# 4) Индекс цен по каждому конкуренту
#    индекс = наша_цена_до_скидки / цена_конкурента
# =========================================================
index_rows = []
for c in COMPETITORS:
    mask = df[c].notna() & df["Цена_до_скидки"].notna() & (df[c] > 0)

    ratios = (df.loc[mask, "Цена_до_скидки"] / df.loc[mask, c]).astype(float)

    index_rows.append({
        "Конкурент": c,
        "Кол-во_SKU_в_сравнении": int(mask.sum()),
        "Индекс_mean": float(ratios.mean()) if len(ratios) else np.nan,
        "Индекс_median": float(ratios.median()) if len(ratios) else np.nan,
        "Доля_SKU_где_мы_дешевле": float((ratios < 1).mean()) if len(ratios) else np.nan,
    })

price_index = pd.DataFrame(index_rows).sort_values("Индекс_mean", na_position="last")

In [5]:
# =========================================================
# 5) Рекомендация: стать минимальными, сохраняя маржу >= 10%
#
# Маржа = (P - C)/P >= 0.10
# => P >= C / (1 - 0.10) = C / 0.9
# Целевая цена под стратегию "минимум рынка": min_comp - 1 руб (или 0)
# Рекомендованная цена = max(целевая_цена, цена_по_марже)
# Рекомендованная скидка = 1 - (рекоменд_цена / цена_до_скидки)
# =========================================================
df["Цена_мин_маржа10"] = df["Цена закупки товара"] / (1 - MIN_MARGIN)
df["Цель_цена_минимум_рынка"] = (df["Мин_цена_конкурента"] - TARGET_DELTA_RUB).clip(lower=0)

df["Можно_быть_минимумом_с_маржой10"] = (
    df["Цена_мин_маржа10"].notna()
    & df["Цель_цена_минимум_рынка"].notna()
    & (df["Цена_мин_маржа10"] <= df["Цель_цена_минимум_рынка"])
)

df["Рекоменд_цена"] = np.maximum(df["Цель_цена_минимум_рынка"], df["Цена_мин_маржа10"])
df["Рекоменд_скидка"] = (1 - (df["Рекоменд_цена"] / df["Цена_до_скидки"])).clip(lower=0)

In [6]:
# =========================================================
# 6) Сводки (для анализа и выводов)
# =========================================================
summary = {
    "SKU_total": int(len(df)),
    "SKU_can_be_min_with_margin10": int(df["Можно_быть_минимумом_с_маржой10"].sum()),
    "SKU_cannot_be_min_with_margin10": int((~df["Можно_быть_минимумом_с_маржой10"]).sum()),
    "Share_can_be_min": float(df["Можно_быть_минимумом_с_маржой10"].mean()),
}

seg_summary = (
    df.groupby("Сегмент", dropna=False)
      .agg(
          SKU=("Код товара", "count"),
          Can_be_min=("Можно_быть_минимумом_с_маржой10", "sum"),
          Avg_recommended_discount=("Рекоменд_скидка", "mean"),
          Median_recommended_discount=("Рекоменд_скидка", "median"),
      )
      .assign(Share_can_be_min=lambda x: x["Can_be_min"] / x["SKU"])
      .reset_index()
      .sort_values(["Share_can_be_min", "SKU"], ascending=[True, False])
)

issue_by_competitor = (
    df.loc[~df["Можно_быть_минимумом_с_маржой10"] & df["Кто_минимум"].notna(), "Кто_минимум"]
      .value_counts()
      .rename_axis("Конкурент_даёт_слишком_низкую_цену")
      .reset_index(name="SKU_count")
)


# =========================================================
# 7) Вывод в ноутбуке (если display нет — будет print)
# =========================================================
print("SUMMARY:", summary)
try:
    display(price_index)
    display(seg_summary)
    display(issue_by_competitor)
except NameError:
    print("\nPRICE INDEX:\n", price_index)
    print("\nSEG SUMMARY:\n", seg_summary)
    print("\nISSUE BY COMPETITOR:\n", issue_by_competitor)


# =========================================================
# 8) Экспорт результатов (для проверки / вложения в репозиторий)
# =========================================================
df.to_csv("task1_item_level.csv", index=False, encoding="utf-8-sig")
price_index.to_csv("task1_price_index.csv", index=False, encoding="utf-8-sig")
seg_summary.to_csv("task1_segment_summary.csv", index=False, encoding="utf-8-sig")
issue_by_competitor.to_csv("task1_issue_by_competitor.csv", index=False, encoding="utf-8-sig")

print("Saved: task1_item_level.csv, task1_price_index.csv, task1_segment_summary.csv, task1_issue_by_competitor.csv")

SUMMARY: {'SKU_total': 683, 'SKU_can_be_min_with_margin10': 525, 'SKU_cannot_be_min_with_margin10': 158, 'Share_can_be_min': 0.7686676427525623}


,Конкурент,Кол-во_SKU_в_сравнении,Индекс_mean,Индекс_median,Доля_SKU_где_мы_дешевле
8,tsum.ru,226,1.006315,1.004649,0.446903
0,letu.ru,193,1.185758,1.100939,0.181347
6,Megamarket,89,1.193348,1.167692,0.247191
5,MagnitMarket,120,1.215558,1.125043,0.208333
7,lamoda.ru,364,1.221289,1.183132,0.107143
9,visagehall.ru,388,1.225810,1.184519,0.012887
2,rivegauche.ru,610,1.226121,1.175971,0.049180
1,iledebeaute.ru,495,1.276763,1.164389,0.064646
4,Wildberries,176,1.295631,1.285739,0.142045
3,Ozon,471,1.318250,1.227816,0.055202


,Сегмент,SKU,Can_be_min,Avg_recommended_discount,Median_recommended_discount,Share_can_be_min
1,Массмаркет,241,88,0.250239,0.300561,0.365145
0,Люкс,442,437,0.209305,0.190425,0.988688


,Конкурент_даёт_слишком_низкую_цену,SKU_count
0,Ozon,84
1,iledebeaute.ru,20
2,Wildberries,15
3,letu.ru,11
4,Megamarket,9
5,rivegauche.ru,7
6,lamoda.ru,7
7,MagnitMarket,4
8,visagehall.ru,1


Saved: task1_item_level.csv, task1_price_index.csv, task1_segment_summary.csv, task1_issue_by_competitor.csv
